In [ ]:
!pip install -q transformers>=4.30.0 accelerate>=0.20.3 soundfile librosa jiwer evaluate tensorboard openpyxl peft

In [ ]:
!pip install git+https://github.com/openai/whisper.git

In [ ]:
!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124

In [ ]:
!pip install datasets==3.6.0

In [ ]:
!pip install --upgrade torchao

In [ ]:
!pip uninstall peft torchao -y
!pip install peft==0.11.0

In [ ]:
import whisper
import torch
import librosa
import numpy as np
from scipy.io import wavfile
import csv
import json
import os
import pandas as pd
import jiwer
import re
from datasets import load_dataset, DatasetDict, features
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    GenerationConfig,
    DataCollatorWithPadding,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate
from dataclasses import dataclass
from typing import Any

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

In [ ]:
model_large = whisper.load_model("large").to(device)

100%|██████████████████████████████████████| 2.88G/2.88G [00:14<00:00, 220MiB/s]


Подгружаем нужные датасеты с гугл диска:

In [ ]:
train_audio = "/content/drive/MyDrive/datasets/train"
validation_audio = "/content/drive/MyDrive/datasets/validation"
test_audio = "/content/drive/MyDrive/datasets/test"

Создаём файлы metadata.csv для каждой выборки. В файле metadata.csv содержится путь к аудиофайлу и предсказанная транскрипция модели. Это делается для того, чтобы не делать эталонные транскрипции в ручную с полного нуля. Такая предварительная авторазметка делает дальнейший процесс немного эффективнее. (Действия я размечала самостоятельно, потому что, во-первых, на тот момент у меня на руках не было таблицы с разметкой, сделанной группой разметчиков, а во-вторых, потому что в этой таблице приведены неполные транскрипции записей, а только сами глаголы, т.е. *пылесосит* вместо *тут мама пылесосит*)

In [ ]:
data = []
for filename in os.listdir(train_audio):
  if filename.endswith('.wav'):
    audio_path = os.path.join(train_audio, filename)
    print(filename)

    result = model_large.transcribe(
        audio_path,
        language="ru",
        task="transcribe",
        fp16=torch.cuda.is_available(),
        temperature=0.0,
        best_of=5,
        beam_size=5,
        patience=2.0,
        compression_ratio_threshold=2.4,
        logprob_threshold=-1.0,
        no_speech_threshold=0.6,
        word_timestamps=True
    )

    if result["segments"] and len(result["segments"][0]["words"]) > 1:
      print(result["segments"][0]["words"][1]["start"] * 1000, result["text"])
    elif result["segments"] and len(result["segments"][0]["words"]) == 1:
      print(result["segments"][0]["words"][0]["start"] * 1000, result["text"])
    else:
      print(result["text"])

    data.append({
        "path": filename,
        "text": result["text"].strip()
    })

df = pd.DataFrame(data)
output_path = os.path.join(train_audio, "train_metadata_ru.csv")
df.to_csv(output_path, index=False, encoding="utf-8")

Обработка: Action_naming_TMS-5-2-2PictureProperties-2.wav
1180.0  Печка пишет.
Обработка: Action_naming_TMS-5-2-3PictureProperties-18.wav
980.0  Вот дочку вырезаю.
Обработка: Action_naming_TMS-5-2-1PictureProperties-23.wav
1460.0  Девочка-чёрт.
Обработка: Action_naming_TMS-5-2-2PictureProperties-13.wav
740.0  Это все будет обниматься.
Обработка: Action_naming_TMS-5-2-3PictureProperties-21.wav
1260.0  Артист поет.
Обработка: Action_naming_TMS-5-2-1PictureProperties-3.wav
880.0  Вот птица летит.
Обработка: Action_naming_TMS-5-2-1PictureProperties-18.wav
1040.0  Это дядя копает.
Обработка: Action_naming_TMS-5-2-1PictureProperties-16.wav
1480.0  Птица Кавказ
Обработка: Action_naming_TMS-5-2-1PictureProperties-1.wav
640.0  И тут мальчик плывет.
Обработка: Action_naming_TMS-5-2-2PictureProperties-12.wav
1260.0  Субтитры сделал DimaTorzok
Обработка: Action_naming_TMS-5-2-1PictureProperties-12.wav
1300.0  Субтитры сделал DimaTorzok
Обработка: Action_naming_TMS-5-2-2PictureProperties-16.wav
900

In [ ]:
data1 = []
for filename in os.listdir(validation_audio):
  if filename.endswith('.wav'):
    audio_path = os.path.join(validation_audio, filename)
    print(filename)

    result = model_large.transcribe(
        audio_path,
        language="ru",
        task="transcribe",
        fp16=torch.cuda.is_available(),
        temperature=0.0,
        best_of=5,
        beam_size=5,
        patience=2.0,
        compression_ratio_threshold=2.4,
        logprob_threshold=-1.0,
        no_speech_threshold=0.6,
        word_timestamps=True
    )

    if result["segments"] and len(result["segments"][0]["words"]) > 1:
      print(result["segments"][0]["words"][1]["start"] * 1000, result["text"])
    elif result["segments"] and len(result["segments"][0]["words"]) == 1:
      print(result["segments"][0]["words"][0]["start"] * 1000, result["text"])
    else:
      print(result["text"])

    data1.append({
        "file_name": filename,
        "transcription": result["text"].strip()
    })

df1 = pd.DataFrame(data1)
output_path = os.path.join(validation_audio, "validation_metadata_ru.csv")
df1.to_csv(output_path, index=False, encoding="utf-8")

Обработка: Action_naming_TMS-25-2-1PictureProperties-11.wav
940.0  Тут девушка открывает.
Обработка: Action_naming_TMS-25-2-1PictureProperties-1.wav
1280.0  Мальчик, привет.
Обработка: Action_naming_TMS-25-2-1PictureProperties-12.wav
1020.0  Вот девушка идёт.
Обработка: Action_naming_TMS-25-2-1PictureProperties-14.wav
800.0  Субтитры создавал DimaTorzok
Обработка: Action_naming_TMS-25-2-1PictureProperties-13.wav
820.0  Тут брат собирает.
Обработка: Action_naming_TMS-25-2-1PictureProperties-10.wav
880.0  Тут мало стирает.
Обработка: Action_naming_TMS-25-2-2PictureProperties-1.wav
800.0  Как мальчик кидает.
Обработка: Action_naming_TMS-25-2-3PictureProperties-33.wav
900.0  Тут девушка идет.
Обработка: Action_naming_TMS-25-2-1PictureProperties-9.wav
760.0  Тут пожарные туши.
Обработка: Action_naming_TMS-25-2-2PictureProperties-26.wav
840.0  Тут точка была есть.
Обработка: Action_naming_TMS-25-2-3PictureProperties-22.wav
820.0  Что-то мама говорит.
Обработка: Action_naming_TMS-25-2-1Pictur

In [ ]:
data2 = []
for filename in os.listdir(test_audio):
  if filename.endswith('.wav'):
    audio_path = os.path.join(test_audio, filename)
    print(filename)

    result = model_large.transcribe(
        audio_path,
        language="ru",
        task="transcribe",
        fp16=torch.cuda.is_available(),
        temperature=0.0,
        best_of=5,
        beam_size=5,
        patience=2.0,
        compression_ratio_threshold=2.4,
        logprob_threshold=-1.0,
        no_speech_threshold=0.6,
        word_timestamps=True
    )

    if result["segments"] and len(result["segments"][0]["words"]) > 1:
      print(result["segments"][0]["words"][1]["start"] * 1000, result["text"])
    elif result["segments"] and len(result["segments"][0]["words"]) == 1:
      print(result["segments"][0]["words"][0]["start"] * 1000, result["text"])
    else:
      print(result["text"])

    data2.append({
        "file_name": filename,
        "transcription": result["text"].strip()
    })

df2 = pd.DataFrame(data2)
output_path = os.path.join(test_audio, "test_metadata1.csv")
df2.to_csv(output_path, index=False, encoding="utf-8")

Обработка: Action_naming_TMS-28-2-1PictureProperties-12.wav
1160.0  Субтитры сделал DimaTorzok
Обработка: Action_naming_TMS-28-2-1PictureProperties-16.wav
1060.0  Субтитры сделал DimaTorzok
Обработка: Action_naming_TMS-28-2-1PictureProperties-10.wav
940.0  Субтитры сделал DimaTorzok
Обработка: Action_naming_TMS-28-2-1PictureProperties-15.wav
1060.0  рабочая сверху
Обработка: Action_naming_TMS-28-2-1PictureProperties-17.wav
1200.0  Мужчина курит.
Обработка: Action_naming_TMS-28-2-1PictureProperties-13.wav
800.0  как девочка
Обработка: Action_naming_TMS-28-2-1PictureProperties-1.wav
740.0  тут мальчик плывет
Обработка: Action_naming_TMS-28-2-1PictureProperties-14.wav
760.0  Вот девочка и нюхает.
Обработка: Action_naming_TMS-28-2-1PictureProperties-11.wav
900.0  Вот так вот.
Обработка: Action_naming_TMS-28-2-2PictureProperties-12.wav
1260.0  вписывали тент
Обработка: Action_naming_TMS-28-2-2PictureProperties-14.wav
860.0  Субтитры сделал DimaTorzok
Обработка: Action_naming_TMS-28-2-3Pictu

Теперь находим бенчмарки для тестовой выборки:

In [ ]:
test_metadata = os.path.join(test_audio, "metadata.csv")
test_metadata_df = pd.read_csv(test_metadata)

data3 = []

for i, row in test_metadata_df.iterrows():
    filename = row["file_name"]

    audio_path = os.path.join(test_audio, filename)

    print(filename)
    result = model_large.transcribe(
        audio_path,
        language="ru",
        task="transcribe",
        fp16=torch.cuda.is_available(),
        temperature=0.4,
        best_of=10,
        beam_size=5,
        patience=2.0,
        compression_ratio_threshold=2.4,
        logprob_threshold=-1.0,
        no_speech_threshold=0.6
    )

    print(result["text"])

    data3.append({
        "file_name": filename,
        "transcription": result["text"].strip()
    })

df3 = pd.DataFrame(data3)
output_path = os.path.join(test_audio, "test_metadata_ru_benchmark1.csv")
df3.to_csv(output_path, index=False, encoding="utf-8")

Action_naming_TMS-28-2-1PictureProperties-12.wav
 Продолжение следует...
Action_naming_TMS-28-2-1PictureProperties-16.wav
 Субтитры сделал DimaTorzok
Action_naming_TMS-28-2-1PictureProperties-10.wav
 Вот это чудо.
Action_naming_TMS-28-2-1PictureProperties-15.wav
 рабочая сверху
Action_naming_TMS-28-2-1PictureProperties-17.wav
 Субтитры сделал DimaTorzok
Action_naming_TMS-28-2-1PictureProperties-13.wav
 как девочка
Action_naming_TMS-28-2-1PictureProperties-1.wav
 тут мальчик плавает
Action_naming_TMS-28-2-1PictureProperties-14.wav
 Вот девочка и нюхает.
Action_naming_TMS-28-2-1PictureProperties-11.wav
 Вот так вот.
Action_naming_TMS-28-2-2PictureProperties-12.wav
 вписывали тент
Action_naming_TMS-28-2-2PictureProperties-14.wav
 Продолжение следует...
Action_naming_TMS-28-2-3PictureProperties-31.wav
 Девушка несет.
Action_naming_TMS-28-2-3PictureProperties-25.wav
 Субтитры сделал DimaTorzok
Action_naming_TMS-28-2-3PictureProperties-23.wav

Action_naming_TMS-28-2-1PictureProperties-22.wav

Склеиваем предсказанные транскрипции и эталонные транскрипции для тестовой выборки в одном файле и загружаем его:

In [ ]:
benchmark_url = "https://docs.google.com/spreadsheets/d/e/2PACX-1vTGvoBF7Z8n6PYvB3Hv0yqM46gu3JstIRU3W24_4Gnw1IiOdfbzNDi3xanJ0X3nKEfb8MDLKAEhzBEE/pub?gid=1971389261&single=true&output=csv"

In [ ]:
test_metadata = pd.read_csv(benchmark_url)
test_metadata['transcription'] = test_metadata['transcription'].fillna('')
print(test_metadata.head())

                                          file_name  \
0  Action_naming_TMS-28-2-1PictureProperties-12.wav   
1  Action_naming_TMS-28-2-1PictureProperties-16.wav   
2  Action_naming_TMS-28-2-1PictureProperties-10.wav   
3  Action_naming_TMS-28-2-1PictureProperties-15.wav   
4  Action_naming_TMS-28-2-1PictureProperties-17.wav   

                transcription correct_transcription  
0      Продолжение следует...         Тут тётя дует  
1  Субтитры сделал DimaTorzok      Тут сын украшает  
2               Вот это чудо.        Тут дядя чинит  
3              рабочая сверху       Рабочий сверлит  
4  Субтитры сделал DimaTorzok     Тут мужчина курит  


In [ ]:
predicted = list(test_metadata["transcription"])
correct = list(test_metadata["correct_transcription"])

print(predicted[:10])
print(correct[:10])

['Продолжение следует...', 'Субтитры сделал DimaTorzok', 'Вот это чудо.', 'рабочая сверху', 'Субтитры сделал DimaTorzok', 'как девочка', 'тут мальчик плавает', 'Вот девочка и нюхает.', 'Вот так вот.', 'вписывали тент']
['Тут тётя дует', 'Тут сын украшает', 'Тут дядя чинит', 'Рабочий сверлит', 'Тут мужчина курит', 'Тут девочка рвёт', 'Тут мальчик плывет', 'Тут девочка нюхает', 'Это капает', 'Тут птица вылетает']


In [ ]:
def clean(text):
  text = text.lower()
  text = re.sub(r'[^\w\s]', '', text)
  text = re.sub(r'([а-яё])\1{2,}', r'\1', text)
  text = re.sub(r'( )\1{2,}', r'\1', text)
  text = re.sub(r'((ч|д)а(ч|д)а)+', '', text)
  text = re.sub(r'(субтитры субтитры)+', '', text)
  text = text.replace('ё', 'е')
  text = text.strip()
  text = text.replace('субтитры сделал dimatorzok', '')
  text = text.replace('субтитры создавал dimatorzok', '')
  text = text.replace('субтитры создал dimatorzok', '')
  text = text.replace('продолжение следует', '')
  text = text.replace('смотрите продолжение в следующей серии', '')
  text = text.replace('субтитры подогнал симон', '')
  text = text.replace('редактор субтитров асемкин корректор аегорова', '')
  text = text.replace('спасибо за просмотр', '')
  text = text.replace('спасибо за внимание', '')
  text = text.replace('продолжаем', '')
  text = text.replace('добро пожаловать', '')
  text = text.replace('дмитрий шепеллетов', '')
  text = text.replace('подпишись на канал и подписывайтесь на наш канал', '')
  text = text.replace('подпишись', '')
  text = text.replace('добавил субтитры dimatorzok', '')
  text = text.replace('субтитры подогнал игорь негода', '')
  text = text.replace('спасибо', '')
  text = text.replace('спасибо за субтитры алексею дубровскому', '')
  text = text.replace('субтитры делал dimatorzok', '')
  text = text.replace('редактор субтитров асемкин', '')
  text = text.replace('субтитры подогнал dimatorzok', '')
  return text

In [ ]:
predicted = list(map(clean, predicted))
print(predicted[:100])

['', '', 'вот это чудо', 'рабочая сверху', '', 'как девочка', 'тут мальчик плавает', 'вот девочка и нюхает', 'вот так вот', 'вписывали тент', '', 'девушка несет', '', '', 'девушка несет', '', '', '', 'уже не поднимает', 'судья свистит', '', '', 'брат плачет', 'пожарная туша', '', '', '', '', 'кто только пишет', 'сейчас запечатаю', '', 'тут точка чистая', '', '', 'пожарная туша', 'матрас приклеен', 'рабочий этаж', 'тут девушка отрывает', '', '', '', 'вот старушка дует', 'весер точит', '', '', '', '', '', '', 'гдето по полке', '', 'вот птица летит', 'девочка льет', 'вот водитель толкает', 'тут все отслеживается', '', '', '', 'пара танцует', '', 'художник рисует', 'смотрите летает', '', 'девушка слушает', '', '', '', '', 'тут папа зажигает', '', '', 'стучится', '', '', 'артист поет', '', '', '', 'работает', 'мама', 'поехали', 'игра собирается', '', '', '', 'вот мальчик клеит', 'и девушку слушать', 'вот на лыжах падает', '', '', '', '', '', '', '', 'и дочка рисует', '', 'рабочий пилит', 'н

In [ ]:
correct = list(map(clean, correct))
print(correct[:100])

['тут тетя дует', 'тут сын украшает', 'тут дядя чинит', 'рабочий сверлит', 'тут мужчина курит', 'тут девочка рвет', 'тут мальчик плывет', 'тут девочка нюхает', 'это капает', 'тут птица вылетает', 'тут дядя моет', 'тут девушка несет', 'тут гость звонит', 'тут дядя моет', 'тут девушка несет', 'тут мальчик надувает', 'тут птица летит', 'тут жених обнимает', 'тут жених обнимает', 'тут судья свистит', 'тут дочка гладит', 'тут костер горит', 'тут брат плачет', 'пожарный тушит', 'тут девочка ест', 'тут дочка пишет', 'юноша бреется', 'тут дочка пишет', 'тут дочка пишет', 'тут тетя печатает', 'тут мама пылесосит', 'тут дочка чистит', 'корова мычит', 'тут бабушка мечтает', 'тут пожарный тушит', 'тут  матрос гребет', 'рабочий пашет', 'тут девушка отрывает', 'тут женщина шьет', 'тут женщина пьет', 'тут папа жарит', 'тут старушка доит', 'тут слесарь точит', 'тут старушка поливает', 'ворона каркает', 'тут абы шнют', 'тут сестра болеет', 'тут брат собирает', 'тут дочка вырезает', 'тут дядя копает', '

In [ ]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
benchmark_wer = wer_metric.compute(predictions=predicted, references=correct)
benchmark_cer = cer_metric.compute(predictions=predicted, references=correct)

print(f"benchmark_wer: {benchmark_wer * 100:.1f}%")
print(f"benchmark_cer: {benchmark_cer * 100:.1f}%")

benchmark_wer: 64.9%
benchmark_cer: 51.6%


Сейчас мы посчитали бенчмарки для модели Large с параметрами temperature = 0.4 и best_of = 10, которую мы признали самой успешной без дообучения. Но на всякий случай посчитаем бенчмарк также для Large с дефолтными параметрами:

In [ ]:
benchmark1_url = "https://docs.google.com/spreadsheets/d/e/2PACX-1vSMBA1a0ZcWPYc64QGuy2JxuEod1BV1mYKQMfgnAvhwF18AeS02jUraSHF2Cvs5hw/pub?gid=766841139&single=true&output=csv"

In [ ]:
test_metadata1 = pd.read_csv(benchmark1_url)
test_metadata1['transcription'] = test_metadata1['transcription'].fillna('')
print(test_metadata1.head())

                                          file_name   transcription  \
0  Action_naming_TMS-28-2-1PictureProperties-12.wav                   
1  Action_naming_TMS-28-2-1PictureProperties-16.wav                   
2  Action_naming_TMS-28-2-1PictureProperties-10.wav                   
3  Action_naming_TMS-28-2-1PictureProperties-15.wav  рабочая сверху   
4  Action_naming_TMS-28-2-1PictureProperties-17.wav  Мужчина курит.   

  correct_transcription  
0         Тут тётя дует  
1      Тут сын украшает  
2        Тут дядя чинит  
3       Рабочий сверлит  
4     Тут мужчина курит  


In [ ]:
predicted1 = list(test_metadata1["transcription"])
correct1 = list(test_metadata1["correct_transcription"])

print(predicted1[:10])
print(correct1[:10])

['', '', '', 'рабочая сверху', 'Мужчина курит.', 'как девочка', 'тут мальчик плывет', 'Вот девочка и нюхает.', 'Вот так вот.', 'вписывали тент']
['Тут тётя дует', 'Тут сын украшает', 'Тут дядя чинит', 'Рабочий сверлит', 'Тут мужчина курит', 'Тут девочка рвёт', 'Тут мальчик плывет', 'Тут девочка нюхает', 'Это капает', 'Тут птица вылетает']


In [ ]:
predicted1 = list(map(clean, predicted1))
print(predicted1[:100])

['', '', '', 'рабочая сверху', 'мужчина курит', 'как девочка', 'тут мальчик плывет', 'вот девочка и нюхает', 'вот так вот', 'вписывали тент', '', 'девушка несет', '', '', 'девушка несет', '', '', '', '', 'судья свистит', '', '', 'брат плачет', 'пожарная туша', 'девочка ест', '', '', '', 'ктото только пишет', 'сейчас запечатаю', 'маммапылесосик', 'тут точка чистая', '', 'бабушка мечтает', '', '', 'рабочий этаж', 'тут девушка отрывает', '', '', '', 'вот старушка дует', 'весер точит', '', '', '', '', 'собирает', '', 'гдето так', '', '', 'девочка льет', 'вот водитель толкает', 'тут все отслеживается', '', 'ну как ты споешь', '', 'пара танцует', '', 'художник рисует', 'смотрите летает', 'негативный', 'девушка слушает', '', '', '', '', '', '', '', 'стучится', '', '', 'артист поет', '', '', '', 'работает', '', 'пусть стащит', 'игра собирается', '', '', '', 'вот мальчик клеит', 'ну девушка слушает', '', '', '', '', '', '', 'бой стащится', '', 'и дочка рисует', '', 'рабочий пилит', 'ну женщина 

In [ ]:
correct1 = list(map(clean, correct1))
print(correct1[:100])

['тут тетя дует', 'тут сын украшает', 'тут дядя чинит', 'рабочий сверлит', 'тут мужчина курит', 'тут девочка рвет', 'тут мальчик плывет', 'тут девочка нюхает', 'это капает', 'тут птица вылетает', 'тут дядя моет', 'тут девушка несет', 'тут гость звонит', 'тут дядя моет', 'тут девушка несет', 'тут мальчик надувает', 'тут птица летит', 'тут жених обнимает', 'тут жених обнимает', 'тут судья свистит', 'тут дочка гладит', 'тут костер горит', 'тут брат плачет', 'пожарный тушит', 'тут девочка ест', 'тут дочка пишет', 'юноша бреется', 'тут дочка пишет', 'тут дочка пишет', 'тут тетя печатает', 'тут мама пылесосит', 'тут дочка чистит', 'корова мычит', 'тут бабушка мечтает', 'тут пожарный тушит', 'тут  матрос гребет', 'рабочий пашет', 'тут девушка отрывает', 'тут женщина шьет', 'тут женщина пьет', 'тут папа жарит', 'тут старушка доит', 'тут слесарь точит', 'тут старушка поливает', 'ворона каркает', 'тут абы шнют', 'тут сестра болеет', 'тут брат собирает', 'тут дочка вырезает', 'тут дядя копает', '

In [ ]:
benchmark_wer1 = wer_metric.compute(predictions=predicted1, references=correct1)
benchmark_cer1 = cer_metric.compute(predictions=predicted1, references=correct1)

print(f"benchmark_wer1: {benchmark_wer1 * 100:.1f}%")
print(f"benchmark_cer1: {benchmark_cer1 * 100:.1f}%")

benchmark_wer1: 63.9%
benchmark_cer1: 51.3%


Видим, что у деволтной модели показатели чуть лучше.

То, что CER значительно ниже WER, указывает на то, что модель в принципе слышит верные звуки и звуковые сочетания, но не всегда правильно определяет слово целиком, а значит, дообучение имеет смысл.

До этого мы нашли бенчмарки для модели large как для самой успешной по метрикам среди всех размеров, доступных у Whisper. Теперь найдём бенчмарк для small модели, потому что мы планируем проводить на ней дообучение и хотелось бы иметь возможность потом сравнить значения

In [ ]:
model_small = whisper.load_model("small").to(device)

100%|███████████████████████████████████████| 461M/461M [00:08<00:00, 57.9MiB/s]


In [ ]:
test_metadata = os.path.join(test_audio, "metadata.csv")
test_metadata_df = pd.read_csv(test_metadata)

data4 = []

for i, row in test_metadata_df.iterrows():
    filename = row["file_name"]

    audio_path = os.path.join(test_audio, filename)

    print(filename)
    result = model_small.transcribe(
        audio_path,
        language="ru",
        task="transcribe",
        fp16=torch.cuda.is_available(),
        temperature=0.0,
        best_of=5,
        beam_size=5,
        patience=2.0,
        compression_ratio_threshold=2.4,
        logprob_threshold=-1.0,
        no_speech_threshold=0.6
    )

    print(result["text"])

    data4.append({
        "file_name": filename,
        "transcription": result["text"].strip()
    })

df4 = pd.DataFrame(data4)
output_path = os.path.join(test_audio, "test_metadata_ru_benchmark_small.csv")
df4.to_csv(output_path, index=False, encoding="utf-8")

Action_naming_TMS-28-2-1PictureProperties-12.wav

Action_naming_TMS-28-2-1PictureProperties-16.wav
 Кристина Кришина Кристина Кришина
Action_naming_TMS-28-2-1PictureProperties-10.wav
 Удачи
Action_naming_TMS-28-2-1PictureProperties-15.wav
 Работает сверху.
Action_naming_TMS-28-2-1PictureProperties-17.wav
 подношена по ряду
Action_naming_TMS-28-2-1PictureProperties-13.wav
 Как девочка, как девочка
Action_naming_TMS-28-2-1PictureProperties-1.wav
 тут мальчик кто-ли?
Action_naming_TMS-28-2-1PictureProperties-14.wav
 Продолжение следует
Action_naming_TMS-28-2-1PictureProperties-11.wav
 Вот так, как-бы.
Action_naming_TMS-28-2-2PictureProperties-12.wav
 Чисто вылетает
Action_naming_TMS-28-2-2PictureProperties-14.wav

Action_naming_TMS-28-2-3PictureProperties-31.wav
 Поверьте, что нечего.
Action_naming_TMS-28-2-3PictureProperties-25.wav

Action_naming_TMS-28-2-3PictureProperties-23.wav
 Десят мое
Action_naming_TMS-28-2-1PictureProperties-22.wav
 Эта девушка несёт.
Action_naming_TMS-28-2-3Pict

In [ ]:
benchmark2_url = "https://docs.google.com/spreadsheets/d/e/2PACX-1vS9dE9_OAAh5bBI-JajeD9QDJOSEqcXZGZGs_EUWnu8R9XX0epOcHUatkXAaxOksbXZKE-B2LyRM3X_/pub?gid=1164322185&single=true&output=csv"

In [ ]:
test_metadata2 = pd.read_csv(benchmark2_url)
test_metadata2['transcription'] = test_metadata2['transcription'].fillna('')
print(test_metadata2.head())

                                          file_name  \
0  Action_naming_TMS-28-2-1PictureProperties-12.wav   
1  Action_naming_TMS-28-2-1PictureProperties-16.wav   
2  Action_naming_TMS-28-2-1PictureProperties-10.wav   
3  Action_naming_TMS-28-2-1PictureProperties-15.wav   
4  Action_naming_TMS-28-2-1PictureProperties-17.wav   

                       transcription correct_transcription  
0                                            Тут тётя дует  
1  Кристина Кришина Кристина Кришина      Тут сын украшает  
2                              Удачи        Тут дядя чинит  
3                   Работает сверху.       Рабочий сверлит  
4                  подношена по ряду     Тут мужчина курит  


In [ ]:
predicted2 = list(test_metadata2["transcription"])
correct2 = list(test_metadata2["correct_transcription"])

print(predicted2[:10])
print(correct2[:10])

['', 'Кристина Кришина Кристина Кришина', 'Удачи', 'Работает сверху.', 'подношена по ряду', 'Как девочка, как девочка', 'тут мальчик кто-ли?', 'Продолжение следует', 'Вот так, как-бы.', 'Чисто вылетает']
['Тут тётя дует', 'Тут сын украшает', 'Тут дядя чинит', 'Рабочий сверлит', 'Тут мужчина курит', 'Тут девочка рвёт', 'Тут мальчик плывет', 'Тут девочка нюхает', 'Это капает', 'Тут птица вылетает']


In [ ]:
predicted2 = list(map(clean, predicted2))
print(predicted2[:100])

['', 'кристина кришина кристина кришина', 'удачи', 'работает сверху', 'подношена по ряду', 'как девочка как девочка', 'тут мальчик ктоли', '', 'вот так какбы', 'чисто вылетает', '', 'поверьте что нечего', '', 'десят мое', 'эта девушка несет', '', 'ух ты старый фиг сейчас у меня', '', 'ктото никогда не мимает', 'чтото я свистил', 'ну что ж подвали', 'хахаха', 'убрать плачет', 'пожарный тушек', 'девушка я', '', 'теперь мы все приедем', '', 'топлочкой пишем', 'сейчас я печатаю', 'ну ну ну ну ну', 'больше почистить', '', 'бабушка не считает', 'от пожарной куши', '', 'это очень паще', 'девушка отрывает', 'проверьте я на швырке', 'примечательный пьек', 'стоп', 'посторожка давай', 'отсюда это вточит', '', '', 'тихотихотихо', '', 'поправь собирает', 'тихо тихо тихо', 'без упаковки', 'ктото наш', '', 'где уже конь мед', '', 'что это лежит', 'подсматривайся', 'поехали', '', 'пора танцовать', '', 'художник рисует', '', 'нега виктория', 'делать косушек', '', 'я не знаю что это такое', '', '', 'кто

In [ ]:
correct2 = list(map(clean, correct2))
print(correct2[:100])

['тут тетя дует', 'тут сын украшает', 'тут дядя чинит', 'рабочий сверлит', 'тут мужчина курит', 'тут девочка рвет', 'тут мальчик плывет', 'тут девочка нюхает', 'это капает', 'тут птица вылетает', 'тут дядя моет', 'тут девушка несет', 'тут гость звонит', 'тут дядя моет', 'тут девушка несет', 'тут мальчик надувает', 'тут птица летит', 'тут жених обнимает', 'тут жених обнимает', 'тут судья свистит', 'тут дочка гладит', 'тут костер горит', 'тут брат плачет', 'пожарный тушит', 'тут девочка ест', 'тут дочка пишет', 'юноша бреется', 'тут дочка пишет', 'тут дочка пишет', 'тут тетя печатает', 'тут мама пылесосит', 'тут дочка чистит', 'корова мычит', 'тут бабушка мечтает', 'тут пожарный тушит', 'тут  матрос гребет', 'рабочий пашет', 'тут девушка отрывает', 'тут женщина шьет', 'тут женщина пьет', 'тут папа жарит', 'тут старушка доит', 'тут слесарь точит', 'тут старушка поливает', 'ворона каркает', 'тут абы шнют', 'тут сестра болеет', 'тут брат собирает', 'тут дочка вырезает', 'тут дядя копает', '

In [ ]:
benchmark_wer2 = wer_metric.compute(predictions=predicted2, references=correct2)
benchmark_cer2 = cer_metric.compute(predictions=predicted2, references=correct2)

print(f"benchmark_wer2: {benchmark_wer2 * 100:.1f}%")
print(f"benchmark_cer2: {benchmark_cer2 * 100:.1f}%")

benchmark_wer2: 96.7%
benchmark_cer2: 66.4%


Приступаем к дообучению

In [ ]:
train_metadata_path = '/content/drive/MyDrive/datasets/train/train_metadata_ru.xlsx'
val_metadata_path = '/content/drive/MyDrive/datasets/validation/validation_metadata_ru.xlsx'
test_metadata_path = '/content/drive/MyDrive/datasets/test/test_metadata_ru.xlsx'

df_train = pd.read_excel(train_metadata_path)
df_val = pd.read_excel(val_metadata_path)
df_test = pd.read_excel(test_metadata_path)

df_train.to_csv('/content/drive/MyDrive/datasets/train/metadata.csv', index=False, encoding='utf-8')
df_val.to_csv('/content/drive/MyDrive/datasets/validation/metadata.csv', index=False, encoding='utf-8')
df_test.to_csv('/content/drive/MyDrive/datasets/test/metadata.csv', index=False, encoding='utf-8')

In [ ]:
all_datasets = load_dataset(
    "audiofolder",
    data_dir="/content/drive/MyDrive/datasets"
)

#здесь для объектов вида AudioDecoder, которые подгрузила функция load_dataset в колонку "audio", подтягивается нормальный аудиофайл,
#переводится в частоту 16кГЦ и сохраняется в память
all_datasets = all_datasets.cast_column("audio", features.Audio(sampling_rate=16000))

print(all_datasets)

Resolving data files:   0%|          | 0/1917 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/253 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/488 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['audio', 'transcription'],
        num_rows: 1914
    })
    validation: Dataset({
        features: ['audio', 'transcription'],
        num_rows: 250
    })
    test: Dataset({
        features: ['audio', 'transcription'],
        num_rows: 485
    })
})


In [ ]:
all_datasets["train"].features

{'audio': Audio(sampling_rate=16000, mono=True, decode=True, id=None),
 'transcription': Value(dtype='string', id=None)}

По формату теперь всё хорошо

In [ ]:
processor = WhisperProcessor.from_pretrained(
    "openai/whisper-small",
    language="russian",
    task="transcribe"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [ ]:
def prepare_dataset(example):
    audio = example["audio"]

    example["input_features"] = processor(     #передаются спектрограммы аудиозаписей
        audio=audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_tensors="pt"                    #возвращает тензоры PyTorch (что удобнее, чем numpy массивы)
    ).input_features[0]                        #берём первый пример из батча (представляет из себя тензор - цифровой представление спектрограммы)

    example["labels"] = processor.tokenizer(      #передаются токены эталонных транскрипций
        example["transcription"],
        return_tensors="pt"
    ).input_ids[0]                            #то же самое: берём первый пример из батча, просто у токенизатора другое название поля для этого

    return example

In [ ]:
all_datasets = all_datasets.map(
    prepare_dataset,
    remove_columns=all_datasets["train"].column_names,
    num_proc=1   #последовательная обработка данных, на одном ядре процессора
)

Map:   0%|          | 0/1914 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

Map:   0%|          | 0/485 [00:00<?, ? examples/s]

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(    #на месте паддинга (пустых элементов) ставим значение -100, чтобы они не учитывались в функции потерь
            labels_batch.attention_mask.ne(1), -100
        )

        batch["labels"] = labels

        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [ ]:
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model.generation_config.language = "russian"
model.generation_config.task = "transcribe"

lora_config = LoraConfig(
    r=32,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

trainable params: 3,538,944 || all params: 245,273,856 || trainable%: 1.4429


In [ ]:
def compute_metrics(pred):

    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    pred_str = [clean(p) for p in pred_str]
    label_str = [clean(l) for l in label_str]

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer, "cer": cer}

In [ ]:
gen_config = GenerationConfig.from_pretrained(      #дублируем ещё раз загрузку с параметрами, чтобы точно не забыл, какой язык нас интересует
    "openai/whisper-small",
    language="russian",
    task="transcribe"
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-finetuned",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    warmup_steps=50,
    num_train_epochs=5,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    save_total_limit=3,
    predict_with_generate=True,
    generation_max_length=225,
    fp16=True,
    report_to=["tensorboard"],
    generation_config=gen_config
)

In [ ]:
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=all_datasets["train"],
    eval_dataset=all_datasets["validation"],
    data_collator=data_collator,
    processing_class=processor,
    compute_metrics=compute_metrics
)

trainer.train()

Step,Training Loss,Validation Loss,Wer,Cer
200,2.617378,2.035661,0.696891,0.415378
400,1.518441,1.324536,0.420984,0.254906
600,1.267506,1.188728,0.357513,0.213577
800,1.222080,1.136594,0.329016,0.198338
1000,1.164555,1.112780,0.319948,0.191642
1200,1.156928,1.104949,0.317358,0.190949


TrainOutput(global_step=1200, training_loss=1.5975255330403646, metrics={'train_runtime': 2724.5525, 'train_samples_per_second': 3.513, 'train_steps_per_second': 0.44, 'total_flos': 2.8105317605376e+18, 'train_loss': 1.5975255330403646, 'epoch': 5.0})

In [ ]:
trainer.state.best_model_checkpoint

'./whisper-small-finetuned/checkpoint-1200'

In [ ]:
trainer.save_model("./whisper-small-finetuned1")
processor.save_pretrained("./whisper-small-finetuned1")

['./whisper-small-finetuned1/processor_config.json']

In [ ]:
test_results = trainer.evaluate(all_datasets["test"])
print(f"Test WER: {test_results['eval_wer'] * 100:.2f}%")
print(f"Test CER: {test_results['eval_cer'] * 100:.2f}%")

Test WER: 39.61%
Test CER: 23.43%


In [ ]:
merged_model = model.merge_and_unload()
merged_model.save_pretrained("/content/drive/MyDrive/whisper-small-final1")
processor.save_pretrained("/content/drive/MyDrive/whisper-small-final1")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['/content/drive/MyDrive/whisper-small-final1/processor_config.json']

In [ ]:
test_samples = all_datasets["test"].select(range(min(20, len(all_datasets["test"]))))
predictions = trainer.predict(test_samples)

pred_str = processor.batch_decode(predictions.predictions, skip_special_tokens=True)
label_str = processor.batch_decode(predictions.label_ids, skip_special_tokens=True)

pred_norm = [clean(p) for p in pred_str]
label_norm = [clean(l) for l in label_str]

for i in range(len(pred_norm)):
    print(f"Пример {i+1}:")
    print(f"  Эталон: '{label_norm[i]}'")
    print(f"  Предсказание: '{pred_norm[i]}'")
    print()

Пример 1:
  Эталон: 'тут тетя дует'
  Предсказание: 'тут тетя твое'

Пример 2:
  Эталон: 'тут сын украшает'
  Предсказание: 'тут тона крышает'

Пример 3:
  Эталон: 'тут дядя чинит'
  Предсказание: 'тут дает отчин'

Пример 4:
  Эталон: 'рабочий сверлит'
  Предсказание: 'тут рабочий сверху'

Пример 5:
  Эталон: 'тут мужчина курит'
  Предсказание: 'тут мужчина курит'

Пример 6:
  Эталон: 'тут девочка рвет'
  Предсказание: 'тут девочка ждет'

Пример 7:
  Эталон: 'тут мальчик плывет'
  Предсказание: 'тут мальчик туляет'

Пример 8:
  Эталон: 'тут девочка нюхает'
  Предсказание: 'тут девочка и нотка'

Пример 9:
  Эталон: 'это капает'
  Предсказание: 'тут лата капает'

Пример 10:
  Эталон: 'тут птица вылетает'
  Предсказание: 'тут птица вылетает'

Пример 11:
  Эталон: 'тут дядя моет'
  Предсказание: 'тут датер моет'

Пример 12:
  Эталон: 'тут девушка несет'
  Предсказание: 'тут девушка несет'

Пример 13:
  Эталон: 'тут гость звонит'
  Предсказание: 'тут твой звонит'

Пример 14:
  Эталон: 'тут 

Тестирование на другом материале (аудио с называнием объектов):

In [ ]:
finetuned_model_path = "/content/drive/MyDrive/whisper-small-final1"
finetuned_model = WhisperForConditionalGeneration.from_pretrained(finetuned_model_path)
processor = WhisperProcessor.from_pretrained(finetuned_model_path)

NameError: name 'WhisperForConditionalGeneration' is not defined

In [ ]:
def transcribe_audio(audio_path, model, processor, language="ru"):

    audio, sample_rate = librosa.load(audio_path, sr=16000)

    input_features = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features

    input_features = input_features.to(model.device)

    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            language=language,
            task="transcribe",
            num_beams=5,
            temperature=0.0,
            max_length=448
        )

    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    return transcription

In [ ]:
finetuned_model = finetuned_model.to(device)
finetuned_model.eval()

WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 768)
      (layers): ModuleList(
        (0-11): 12 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (f

In [ ]:
audios = ["/content/Object_naming_TMS -17-2-1PictureProperties-1.wav", "/content/Object_naming_TMS -17-2-1PictureProperties-2.wav",
          "/content/Object_naming_TMS -17-2-1PictureProperties-3.wav", "/content/Object_naming_TMS -17-2-1PictureProperties-4.wav",
          "/content/Object_naming_TMS -17-2-1PictureProperties-5.wav", "/content/Object_naming_TMS -17-2-1PictureProperties-6.wav",
          "/content/Object_naming_TMS -17-2-1PictureProperties-7.wav", "/content/Object_naming_TMS -17-2-1PictureProperties-8.wav",
          "/content/Object_naming_TMS -17-2-1PictureProperties-9.wav", "/content/Object_naming_TMS -17-2-1PictureProperties-10.wav",
          "/content/Object_naming_TMS -17-2-1PictureProperties-11.wav", "/content/Object_naming_TMS -17-2-1PictureProperties-12.wav",
          "/content/Object_naming_TMS -17-2-1PictureProperties-13.wav", "/content/Object_naming_TMS -17-2-1PictureProperties-14.wav",
          "/content/Object_naming_TMS -17-2-1PictureProperties-15.wav", "/content/Object_naming_TMS -17-2-1PictureProperties-16.wav",
          "/content/Object_naming_TMS -17-2-1PictureProperties-17.wav", "/content/Object_naming_TMS -17-2-1PictureProperties-18.wav",
          "/content/Object_naming_TMS -17-2-1PictureProperties-19.wav", "/content/Object_naming_TMS -17-2-1PictureProperties-20.wav",
          "/content/Object_naming_TMS -17-2-1PictureProperties-21.wav", "/content/Object_naming_TMS -17-2-1PictureProperties-22.wav",
          "/content/Object_naming_TMS -17-2-1PictureProperties-23.wav", "/content/Object_naming_TMS -17-2-1PictureProperties-24.wav",
          "/content/Object_naming_TMS -17-2-1PictureProperties-25.wav", "/content/Object_naming_TMS -17-2-1PictureProperties-26.wav",]

In [ ]:
for audio in audios:
  print(transcribe_audio(audio, finetuned_model, processor, language="ru"))

Тут третон автобус.
 Эта бабочка.
 Это дилка. Я вас не прошу попрыгнуть в минуту.
Татьякин поворот.
Тут тихий рак.
Тут шок.
Тут вода падает.
Тут педоситок порт.
Тут панка.
Тут это кровать.
Тут лампа
Тут черепаха
Тут голова.
Тут ракетка.
Тут этот лицо
Тут патчилл
Тут пушка
 Это щатка.
Тут лужина.
Тут лоб.
Тут левый паспорт.
Та это букет.
Тут папочка
Тут осталось.
Тут котилка.
 Это пастыра.


In [ ]:
model_large = whisper.load_model("large").to(device)

100%|█████████████████████████████████████| 2.88G/2.88G [00:44<00:00, 68.8MiB/s]


In [ ]:
for audio in audios:
  result = model_large.transcribe(
      audio,
      language="ru",
      task="transcribe",
      fp16=torch.cuda.is_available(),
      temperature=0.0,
      best_of=5,
      beam_size=5,
      patience=2.0,
      compression_ratio_threshold=2.4,
      logprob_threshold=-1.0,
      no_speech_threshold=0.6,
      word_timestamps=True
  )

  if result["segments"] and len(result["segments"][0]["words"]) > 1:
    print(result["segments"][0]["words"][1]["start"] * 1000, result["text"])
  elif result["segments"] and len(result["segments"][0]["words"]) == 1:
    print(result["segments"][0]["words"][0]["start"] * 1000, result["text"])
  else:
    print(result["text"])

540.0  Это автобус.
520.0  Это бабочка.
3480.0  Продолжение следует...
680.0  Продолжение следует...
620.0  Продолжение следует...
520.0  Это мешок.
800.0  Субтитры сделал DimaTorzok
420.0  Я не считаю форм.
480.0  Это банка.
560.0  Это кровать.
580.0  Это лампа.
540.0  Это черепаха.
540.0  Это грамотно.
520.0  Это ракета.
520.0  Это колесо.
680.0  Субтитры сделал DimaTorzok
1320.0  Субтитры сделал DimaTorzok
380.0  Я не счастлив.
560.0  Это поведение.
3560.0  Субтитры сделал DimaTorzok
3660.0  Субтитры сделал DimaTorzok
500.0  Это букет.
540.0  Это башка.
620.0  Субтитры сделал DimaTorzok
600.0  Это копилка.
880.0  Продолжение следует...


Кажется, что Large модель без дообучения на записях с объектами справялется лучше, чем дообученная на действиях small модель

Большие размеры Whisper (medium, large) дообучить на ту же задачу не получилось из-за нехватки вычислительных ресурсов